# TP2 - Etapa 2 en Google Colab: dataset, preprocesamiento, entrenamiento y comparacion de modelos

Notebook de trabajo de la Etapa 2: analisis del dataset, preprocesamiento, fine-tuning de
ResNet18 (Modelo A, obligatorio), CNN propia (Modelo B, obligatorio) y estudio comparativo.

El entrenamiento se realiza en Colab para aprovechar la GPU:
**Entorno de ejecucion -> Cambiar tipo de entorno de ejecucion -> GPU (T4)**.

Requisitos previos (en tu fork):
- `train_classifier`, `evaluate_classifier` y `extract_custom_embedding` implementadas
  en `src/lib/services/classifier_service.py` (esta notebook solo las orquesta).

Al finalizar:
- Descargar los checkpoints generados y colocarlos en `models/` de tu entorno local
  (la aplicacion los usa en las pestañas Etapa 1 y 2 del frontend).
- Publicarlos en un link de solo lectura publico para los docentes.
- Entregar esta notebook ejecutada, con sus salidas.

## Equipo
- Alumno 1: Gianfranco Frattini
- Alumno 2: Matías Prado

## 1. Clonar el repositorio

Si tu fork es privado, genera un token de acceso (GitHub -> Settings -> Developer settings ->
Personal access tokens) y usalo en la URL, o sube un zip del proyecto a Colab/Drive.

In [ ]:
# TO-DO: completar con la URL de tu fork.
# Repo privado: https://<TOKEN>@github.com/<usuario>/tuia-dog-recognition-app.git
REPO_URL = "https://github.com/GianfrancoFrattini/tuia-dog-recognition-app.git"

!git clone $REPO_URL proyecto
%cd proyecto

## 2. Instalar dependencias

Colab ya incluye torch, torchvision, opencv, numpy, scikit-learn y matplotlib;
solo se instala lo que falta.

In [ ]:
!pip install -q pydantic-settings python-dotenv kagglehub albumentations

## 3. Configuracion del entorno

En Colab no hay PostgreSQL; la Etapa 2 no usa la base vectorial, por lo que se desactiva
pgvector. La configuracion se define por variables de entorno **antes** de importar `lib`.

In [ ]:
import os
import sys
from pathlib import Path

os.environ["USE_PGVECTOR"] = "false"
# Ajusta aca cualquier otra variable (IMAGE_SIZE, EMBEDDING_DIM, paths, etc.)

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from lib.config import settings

import torch
print("dataset_path:", settings.dataset_path)
print("GPU disponible:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

## 4. Descargar el dataset

In [ ]:
!python scripts/download_dataset.py
!ls data/dataset

## 5. Analisis del dataset

El dataset **70 Dog Breeds Image Dataset** ya viene separado en tres splits:
`train/`, `valid/` y `test/`, cada uno con una subcarpeta por raza.

A continuacion se analiza:
- La distribucion de clases (cantidad de imagenes por raza) en cada split.
- El balance general del dataset.
- Tamano de imagenes (para justificar el resize).
- Un conjunto independiente para evaluacion (imagenes descargadas de internet).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from PIL import Image

dataset_path = Path(settings.dataset_path)

def count_per_breed(split):
    root = dataset_path / split
    counts = {}
    for breed_dir in sorted(root.iterdir()):
        if breed_dir.is_dir():
            n = sum(1 for f in breed_dir.iterdir() if f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp'})
            counts[breed_dir.name] = n
    return counts

train_counts = count_per_breed('train')
valid_counts = count_per_breed('valid')
test_counts = count_per_breed('test')

df = pd.DataFrame({'train': train_counts, 'valid': valid_counts, 'test': test_counts}).fillna(0).astype(int)
df['total'] = df.sum(axis=1)
df = df.sort_values('total', ascending=False)

print(f"Razas totales: {len(df)}")
print(f"Imagenes totales: {df['total'].sum()} (train={df['train'].sum()}, valid={df['valid'].sum()}, test={df['test'].sum()})")
print(f"\nTop 5 razas con mas imagenes:")
print(df.head(5))
print(f"\nBottom 5 razas con menos imagenes:")
print(df.tail(5))
print(f"\nRango por raza: min={df['total'].min()}, max={df['total'].max()}, media={df['total'].mean():.1f}")

fig, ax = plt.subplots(figsize=(16, 5))
df[['train', 'valid', 'test']].plot(kind='bar', stacked=True, ax=ax)
ax.set_title('Distribucion de imagenes por raza (train/valid/test)')
ax.set_xlabel('Raza')
ax.set_ylabel('Cantidad de imagenes')
ax.set_xticklabels([])
plt.tight_layout()
plt.show()

In [ ]:
# Analisis del tamano de las imagenes (muestra del split train)
sizes = []
train_root = dataset_path / 'train'
for breed_dir in sorted(train_root.iterdir())[:10]:
    for img_path in list(breed_dir.iterdir())[:10]:
        if img_path.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
            with Image.open(img_path) as im:
                sizes.append(im.size)  # (w, h)

ws, hs = zip(*sizes)
print(f"Muestra de {len(sizes)} imagenes:")
print(f"Ancho: min={min(ws)}, max={max(ws)}, media={sum(ws)/len(ws):.0f}")
print(f"Alto:  min={min(hs)}, max={max(hs)}, media={sum(hs)/len(hs):.0f}")
print("\nJustificacion del resize a 224x224: las imagenes tienen tamanos variables,")
print("por lo que se redimensionan a 224x224 (tamano estandar de ResNet/ImageNet).")

In [ ]:
# Conjunto independiente para evaluacion.
# Se construye con imagenes descargadas de internet (fuera del dataset de Kaggle)
# colocadas en data/external/<raza>/. Si no existe, se reporta como TO-DO manual.
external_root = dataset_path.parent / 'external'
if external_root.is_dir():
    ext_counts = count_per_breed_in(external_root)
    print(f"Conjunto independiente: {sum(ext_counts.values())} imagenes en {len(ext_counts)} razas")
else:
    print(f"Conjunto independiente: ubicarlo en {external_root} (imagenes de internet).")
    print("Por ahora se evalua con el split test/ provisto por el dataset.")

def count_per_breed_in(root):
    counts = {}
    for breed_dir in sorted(root.iterdir()):
        if breed_dir.is_dir():
            n = sum(1 for f in breed_dir.iterdir() if f.suffix.lower() in {'.jpg', '.jpeg', '.png'})
            counts[breed_dir.name] = n
    return counts

## 6. Preprocesamiento

Tecnicas aplicadas y justificadas:

| Tecnica | Valor | Justificacion |
|---|---|---|
| **Resize** | 224x224 | Tamano estandar de ResNet/ImageNet; las imagenes tienen tamanos variables. |
| **Normalizacion** | mean/std ImageNet | El modelo pre-entrenado espera esta distribucion de entrada. |
| **Horizontal Flip** | p=0.5 | El perro no cambia de raza al espejar; duplica efectivamente la data. |
| **Rotacion** | +/-15 grados | Tolerancia a inclinaciones de la camara/foto. |
| **Blur (GaussianBlur)** | p=0.3, k=3 | Simula desenfoque y robustez a baja calidad. |
| **Variacion de brillo** | 0.2 | Compensa condiciones de iluminacion variable. |
| **Variacion de contraste** | 0.2 | Idem brillo, mejora generalizacion. |
| **Ruido** | via ColorJitter + Blur | Se cubre con las variaciones anteriores. |
| **RandomResizedCrop** | scale 0.8-1.0 | Simula distintos encuadres y distancias. |
| **Filtrado de baja calidad** | manual | Se descartan imagenes corruptas/no-legibles en el EDA. |

El data augmentation **solo** se aplica al split de train; valid/test usan solo resize+normalizacion.

In [ ]:
import torchvision.transforms as T

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
SIZE = settings.image_size

train_tf = T.Compose([
    T.RandomResizedCrop(SIZE, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=0.3),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = T.Compose([
    T.Resize((SIZE, SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Ejemplo visual: aplicar augmentation a una imagen
from torchvision import datasets
sample_ds = datasets.ImageFolder(str(dataset_path / 'train'), transform=train_tf)
img, label = sample_ds[0]
def show_tensor(t, title=''):
    import numpy as np
    img = t.permute(1, 2, 0).numpy()
    img = img * IMAGENET_STD + IMAGENET_MEAN
    img = np.clip(img, 0, 1)
    plt.imshow(img); plt.title(title); plt.axis('off'); plt.show()

print(f"Raza de ejemplo: {sample_ds.classes[label]}")
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax in axes:
    img, _ = sample_ds[0]
    arr = (img.permute(1,2,0).numpy() * IMAGENET_STD + IMAGENET_MEAN).clip(0,1)
    ax.imshow(arr); ax.axis('off')
plt.suptitle('Ejemplos de data augmentation sobre la misma imagen')
plt.tight_layout(); plt.show()

## 7. Modelo A (obligatorio): fine-tuning de ResNet18

`train_classifier` realiza fine-tuning de ResNet18 pre-entrenado en ImageNet,
reemplazando la capa FC final por una lineal con `num_classes` salidas.

**Hiperparametros utilizados:**
- Optimizador: AdamW (lr=1e-3, weight_decay=1e-4)
- Scheduler: ReduceLROnPlateau (patience=2, factor=0.5)
- Batch size: 32
- Epochs: 15
- Loss: CrossEntropy
- Data augmentation: ver seccion 6 (solo train)

El checkpoint se guarda en `models/resnet18_finetuned.pth` con el formato:
`{state_dict, classes, arch, embedding_dim, num_classes, history}`.

In [ ]:
from lib.bootstrap import build_classifier

classifier = build_classifier(settings)
classifier.set_active_model("resnet18_finetuned")
classifier.train_classifier()

In [ ]:
metrics_a = classifier.evaluate_classifier()
metrics_a

### Curvas de entrenamiento y matriz de confusion (Modelo A)

Se leen las curvas de `history` guardadas en el checkpoint y se grafican junto
con la matriz de confusion sobre el conjunto de test.

In [ ]:
import torch, numpy as np
ckpt = torch.load(classifier.active_checkpoint, map_location='cpu', weights_only=False)
history = ckpt.get('history', [])
classes = ckpt.get('classes', [])

# Curvas de loss
if history:
    ep = [h['epoch'] for h in history]
    tl = [h['train_loss'] for h in history]
    vl = [h['val_loss'] for h in history]
    plt.figure(figsize=(8,5))
    plt.plot(ep, tl, label='train_loss')
    plt.plot(ep, vl, label='val_loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Curvas de entrenamiento - ResNet18')
    plt.legend(); plt.grid(True); plt.show()

In [ ]:
# Matriz de confusion (Modelo A) sobre test
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, cls = classifier._load_classifier_model()
model = model.to(device).eval()
tf = transforms.Compose([transforms.Resize((SIZE,SIZE)), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
test_ds = datasets.ImageFolder(str(dataset_path/'test'), transform=tf)
loader = torch.utils.data.DataLoader(test_ds, batch_size=32, shuffle=False)
y_true, y_pred = [], []
with torch.no_grad():
    for imgs, labels in loader:
        out = model(imgs.to(device))
        y_pred.extend(out.argmax(1).cpu().tolist()); y_true.extend(labels.tolist())

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(cls))))
fig, ax = plt.subplots(figsize=(14,12))
ConfusionMatrixDisplay(cm, display_labels=cls).plot(xticks_rotation=90, ax=ax, colorbar=False)
ax.set_title('Matriz de confusion - ResNet18 fine-tuned (test)')
plt.tight_layout(); plt.show()

## 8. Modelo B (obligatorio): CNN propia

`train_classifier` con el modelo activo `cnn_custom` entrena una CNN propia
(bloques conv + BN + ReLU + MaxPool + GAP + FC, embedding de 512-d) desde cero.

**Hiperparametros:** mismos que Modelo A para comparar de forma justa.
Checkpoint en `models/cnn_custom.pth`.

In [ ]:
classifier.set_active_model("cnn_custom")
classifier.train_classifier()
metrics_b = classifier.evaluate_classifier()
metrics_b

In [ ]:
# Curvas y matriz de confusion (Modelo B)
ckpt_b = torch.load(classifier.active_checkpoint, map_location='cpu', weights_only=False)
history_b = ckpt_b.get('history', [])
if history_b:
    ep = [h['epoch'] for h in history_b]; tl = [h['train_loss'] for h in history_b]; vl = [h['val_loss'] for h in history_b]
    plt.figure(figsize=(8,5)); plt.plot(ep, tl, label='train_loss'); plt.plot(ep, vl, label='val_loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Curvas de entrenamiento - CNN custom'); plt.legend(); plt.grid(True); plt.show()

In [ ]:
model_b, cls_b = classifier._load_classifier_model()
model_b = model_b.to(device).eval()
y_true_b, y_pred_b = [], []
with torch.no_grad():
    for imgs, labels in loader:
        out = model_b(imgs.to(device))
        y_pred_b.extend(out.argmax(1).cpu().tolist()); y_true_b.extend(labels.tolist())
cm_b = confusion_matrix(y_true_b, y_pred_b, labels=list(range(len(cls_b))))
fig, ax = plt.subplots(figsize=(14,12))
ConfusionMatrixDisplay(cm_b, display_labels=cls_b).plot(xticks_rotation=90, ax=ax, colorbar=False)
ax.set_title('Matriz de confusion - CNN custom (test)')
plt.tight_layout(); plt.show()

## 9. Estudio comparativo

Comparacion de metricas entre ResNet18 fine-tuned (Modelo A) y CNN custom (Modelo B).

In [ ]:
import pandas as pd
comp = pd.DataFrame([metrics_a, metrics_b], index=['ResNet18 fine-tuned', 'CNN custom']).round(4)
comp

In [ ]:
# Analisis de errores: clases con peor desempeno (Modelo A)
from sklearn.metrics import classification_report
report = classification_report(y_true, y_pred, target_names=cls, output_dict=True, zero_division=0)
rep_df = pd.DataFrame(report).T.drop(columns=['support']).round(3)
worst = rep_df.iloc[:-3].sort_values('f1-score').head(10)
print('Top 10 razas con peor F1 (ResNet18):')
print(worst)

# Comparacion de accuracy global
fig, ax = plt.subplots(figsize=(8,5))
comp[['accuracy','precision','recall','specificity','f1']].plot(kind='bar', ax=ax)
ax.set_title('Comparacion de metricas: ResNet18 vs CNN custom'); ax.set_ylabel('Score')
ax.set_xticklabels(comp.index, rotation=0); plt.tight_layout(); plt.show()

### Conclusiones del estudio comparativo

- **ResNet18 fine-tuned** aprovecha el transfer learning de ImageNet y por ende parte de
  una representacion ya util; se espera mayor accuracy y mejor F1 que la CNN propia.
- **CNN custom** entrena desde cero, por lo que necesita mas datos/epochs para converger;
  es util como baseline de comparacion y para entender el valor del transfer learning.
- **Trade-offs**: ResNet18 tiene ~11M parametros (mayor consumo de memoria/inferencia) vs
  la CNN custom (mas liviana). En GPU (RTX 3080 / T4 Colab) ambos entrenan en minutos.
- Las razas peor clasificadas suelen ser visualmente similares (ej. variantes de terrier o
  spaniel), lo que se refleja en los falsos positivos de la matriz de confusion.
- **Specificity** alta en ambos modelos (clasificacion multiclase con muchas clases 
  verdaderas negativas por muestra).

(Completar con los numeros reales obtenidos tras ejecutar la notebook.)

## 10. Descargar los checkpoints

Descargalos y colocalos en `models/` de tu entorno local para que la aplicacion los use
(pestanas Etapa 1 y 2 del frontend). Recorda publicarlos en un link de solo lectura publico.

In [ ]:
!ls -lh models/

try:
    from google.colab import files
    files.download(str(settings.model_path / settings.resnet18_model_name))
    files.download(str(settings.model_path / settings.cnn_custom_model_name))
except ImportError:
    print("Fuera de Colab: los checkpoints quedan en models/")